# Generate evaluation data for the finetuned and baseline models

In [ ]:
import json
from src import utils, llms


In [ ]:
USE_GOOGLE_COLAB = False

In [ ]:
if USE_GOOGLE_COLAB:
    from google.colab import drive
    import sys
    drive.mount('/content/drive', force_remount=True)
    sys.path.append('/content/drive/MyDrive/cs329x/')

    from google.colab import userdata
    huggingface_token = userdata.get("HUGGINGFACE_TOKEN")

    data_directory = '/content/drive/MyDrive/cs329x'

else:
    import os
    huggingface_token = os.environ['HUGGINGFACE_TOKEN']

    data_directory = './results'

final_model_path = f"{data_directory}/dpo_qwen_final"


In [ ]:
model_name = "Qwen/Qwen3-4B-Instruct-2507"

In [ ]:
# Load both models for evaluation
print("Loading models for evaluation...")

# Load baseline model
baseline_model = llms.HuggingFaceClient(
    model=model_name,
    temperature=0.8,
    load_in_4bit=True
)
print("Baseline model loaded")

# Load fine-tuned model
finetuned_model = llms.HuggingFaceClient(
    model=final_model_path,
    temperature=0.8,
    load_in_4bit=True
)
print("Fine-tuned model loaded")

In [ ]:
# Load documents
documents = utils.load_dataset_from_jsonl(f'{data_directory}/training_docs.jsonl')
print(f"Loaded {len(documents)} documents")

In [ ]:
# Create test set (use documents not in training if available, or use a subset)
# For simplicity, we'll evaluate on the training set
test_scenarios = []
for doc_idx, entry in enumerate(documents):
    thread_len = len(entry.comment_thread)
    # Use middle point of thread for testing
    if thread_len >= 3:
        n_comments = thread_len // 2
        test_scenarios.append({
            'doc_idx': doc_idx,
            'entry': entry,
            'n_comments': n_comments
        })

print(f"Created {len(test_scenarios)} test scenarios")

In [ ]:
from src.dataset_gen import _generate_agent_interventions_for_doc

def generate_intervention_datapoints_local(
    documents,
    agent_llm_1,
    agent_llm_2,
    use_hidden_prompt_for_interventions,
    conversation_points
):
    # Generate paired interventions for each document
    intervention_pairs = []

    for doc_idx, entry in enumerate(documents):
      print(f'starting with {doc_idx}')
      results = _generate_agent_interventions_for_doc(
          doc_idx,
          entry,
          agent_llm_1,
          agent_llm_2,
          use_hidden_prompt_for_interventions,
          conversation_points
      )
      if results:
          intervention_pairs.extend(results)
    return intervention_pairs


In [ ]:
eval_size = 40

interventions_for_eval_data = generate_intervention_datapoints_local(
    documents=documents[:eval_size],
    agent_llm_1=baseline_model,
    agent_llm_2=finetuned_model,
    use_hidden_prompt_for_interventions=False,
    conversation_points=[2, 3, 4]
)

print(f"\n\nGenerated {len(interventions_for_eval_data)} training examples")

In [ ]:
# Example: saving a text file
with open('/content/drive/MyDrive/cs329x/eval_generations_all_type_combos.jsonl', 'w', encoding='utf-8') as f:
    for item in interventions_for_eval_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')